# 📊 Session 2 — RAG Evaluation & AI Quality Hands-On

**Companion to:** `Session2_RAG_Evaluation_v1.pptx`

**What we'll do (2 hours):**
1. **Theory primer** — Why classical NLP metrics (BLEU / ROUGE / METEOR / BERTScore) do **not** work for RAG
2. The **RAG-evaluation landscape** — RAGAS · G-Eval · DeepEval · TruLens · Phoenix · ARES · LangSmith · OpenAI Evals — and how to pick one
3. **LLM-as-Judge** — the technique that powers all modern RAG evaluators. When to use it, when *not* to use it
4. Rebuild Naive + Advanced RAG (from Session 1) — quickly, so eval has something to score
5. **Hands-On #1 — RAGAS** on both pipelines: 4 core metrics (Faithfulness, Answer Relevancy, Context Precision, Context Recall) + tour of the full RAGAS metric catalogue
6. Read the results — per-question drill-down, bar chart, score → fix
7. **Hands-On #2 — Custom G-Eval judge** for a criterion RAGAS doesn't cover (Conciseness)

**Stack:** Groq (Llama-3.1-8B judge) · LangChain · FAISS · BM25 · Cross-Encoder · RAGAS · custom G-Eval · pandas · matplotlib

> ⚠️ **Free Groq tier** ≈ 100 K tok/day on 70B. We default to **`llama-3.1-8b-instant`** as judge — fast, generous quota, safe for live workshops.

---
## 🔑 Prereqs
| Item | How |
|---|---|
| Python | 3.9+ |
| `GROQ_API_KEY` | Free at [console.groq.com/keys](https://console.groq.com/keys) |
| Internet | Downloads model + PDF on first run |


---
## 📚 Theory Primer — Why Classical NLP Metrics Break for RAG

Before we run RAGAS, understand **why** the metrics you may have seen in NLP courses (BLEU, ROUGE, METEOR, BERTScore) are the **wrong tool** for RAG evaluation.

### The 4 classical metrics — what they measure

| Metric | What it does | Origin |
|---|---|---|
| **BLEU** (Bilingual Evaluation Understudy) | Precision of n-gram overlap between candidate and reference | Machine translation (2002) |
| **ROUGE** (Recall-Oriented Understudy for Gisting Evaluation) | Recall of n-gram overlap; ROUGE-L uses longest common subsequence | Summarisation (2004) |
| **METEOR** | Aligns unigrams using stemming + WordNet synonyms, then F-score | Machine translation (2005) |
| **BERTScore** | Cosine similarity between BERT embeddings of candidate & reference tokens | Any NLG (2019) |

All four require a **reference answer** and compare the generated text to it — either at token level (BLEU/ROUGE/METEOR) or at embedding level (BERTScore).

### Why they fail on RAG

RAG produces **open-ended natural-language answers grounded in retrieved documents**. Traditional metrics were built for tasks where a small set of "correct" surface forms exists (translation, headline generation). RAG breaks every assumption they make:

| Problem | Concrete example | Result |
|---|---|---|
| **Paraphrase blindness** | Ground truth: *"The authors are Vaswani et al."*<br>Answer: *"Ashish Vaswani wrote it with 7 co-authors."* | BLEU / ROUGE ≈ 0 — **penalises a correct answer** |
| **Rewards fluent hallucination** | Answer copies exact reference wording but adds a wrong fact | High BLEU / ROUGE — **hides a hallucination** |
| **Can't score `contexts`** | Retrieval quality (which chunks were pulled) has no "reference" | Metric not applicable — **retriever is invisible** |
| **Ignores faithfulness** | Answer includes information *not in the retrieved context* — the essence of hallucination | No signal at all |
| **Ignores question relevance** | An off-topic but well-worded answer | Scored the same as an on-topic one |
| **Reference bias** | Only one "correct" phrasing counted | Real questions have many valid answers |
| **BERTScore is soft-lexical** | Better than BLEU, still just token-level similarity | Doesn't detect claim-level errors |

### The core mismatch

```
     Classical NLP metrics                    What RAG actually needs
     ──────────────────────                    ───────────────────────
   "Did the words match?"                  →  "Is the answer FAITHFUL to the retrieved context?"
   "Same n-grams as reference?"            →  "Is the answer RELEVANT to the question?"
   "How close to gold text?"               →  "Did the retriever find the RIGHT chunks?"
   "Single reference is enough"            →  "Many valid answers per question"
```

RAG needs **reasoning-based judgement**, not surface similarity. That's why the field moved to LLM-as-Judge (RAGAS, G-Eval, DeepEval, TruLens…).

### Rule of thumb

| Task | Best metric family |
|---|---|
| Machine translation to a fixed reference | BLEU / COMET |
| Extractive summarisation with gold summary | ROUGE |
| Classification / QA with a short exact answer | Exact match, F1 |
| **RAG / open-ended Q&A / chatbots** | **LLM-as-Judge (RAGAS / G-Eval / DeepEval)** |
| Reference-free fluency check | Perplexity, BLEURT |

We'll now run a **live demo** proving BLEU/ROUGE break, then map the modern evaluation landscape.


In [ ]:
# ══════════════════════════════════════════════════════════════════
# 🧪 LIVE DEMO — BLEU & ROUGE fail on a correct paraphrase
# (Pure Python, no extra installs — approximations of BLEU-1 / ROUGE-L)
# ══════════════════════════════════════════════════════════════════
from collections import Counter
from difflib import SequenceMatcher

def bleu1_like(candidate: str, reference: str) -> float:
    """Unigram BLEU-style precision: fraction of candidate tokens that appear in reference."""
    cand = candidate.lower().split()
    ref = Counter(reference.lower().split())
    if not cand:
        return 0.0
    overlap = sum(min(cand.count(w), ref[w]) for w in set(cand))
    return round(overlap / len(cand), 3)

def rouge_l_like(candidate: str, reference: str) -> float:
    """ROUGE-L approximation via longest-common-subsequence ratio (F-measure style)."""
    a, b = candidate.lower().split(), reference.lower().split()
    if not a or not b:
        return 0.0
    lcs = SequenceMatcher(None, a, b).find_longest_match(0, len(a), 0, len(b)).size
    p = lcs / len(a); r = lcs / len(b)
    return round(2 * p * r / (p + r), 3) if (p + r) else 0.0

REFERENCE = "Ashish Vaswani, Noam Shazeer, Niki Parmar, and five others wrote the Transformer paper."

candidates = [
    ("✅ Correct paraphrase (different words)",
     "The Attention Is All You Need paper was authored by Vaswani and seven collaborators."),
    ("✅ Correct summary (different phrasing)",
     "It was written by a team of eight researchers including Ashish Vaswani."),
    ("❌ Fluent hallucination (copies reference wording, wrong facts)",
     "Ashish Vaswani, Noam Shazeer, Niki Parmar, and five others wrote the BERT paper."),
    ("❌ Complete lie (with high lexical overlap)",
     "Ashish Vaswani and Noam Shazeer alone wrote the Transformer paper."),
    ("🎯 Word-for-word copy of the reference",
     REFERENCE),
]

print(f"REFERENCE: {REFERENCE}\n")
print(f"{'Candidate':<58}  {'BLEU-1':>7}  {'ROUGE-L':>8}  Verdict")
print("─" * 100)
for label, cand in candidates:
    b = bleu1_like(cand, REFERENCE)
    r = rouge_l_like(cand, REFERENCE)
    print(f"{label:<58}  {b:>7.3f}  {r:>8.3f}  {cand[:60]}…")

print("\n💡 Notice:")
print("   • The two CORRECT paraphrases score low — the metric penalises correct answers")
print("   • The FLUENT HALLUCINATION scores near-perfect — the metric rewards a wrong answer")
print("   • Only word-for-word copies get top scores → useless for open-ended RAG")
print("\n👉 This is why we need LLM-as-Judge metrics (RAGAS, G-Eval, DeepEval).")


---
## 🗺️ The RAG-Evaluation Landscape — RAGAS vs G-Eval vs DeepEval vs …

Because BLEU/ROUGE can't measure faithfulness or retrieval quality, an entire ecosystem of **LLM-as-Judge frameworks** has grown up around RAG. Here's the map.

### The main frameworks

| Framework | Style | Best at | Notes |
|---|---|---|---|
| **RAGAS** | Task-specific metric library (Faithfulness, Answer Relevancy, Context Precision/Recall, Answer Correctness, Context Entities Recall, Noise Sensitivity) | End-to-end RAG scoring — the *de facto* default | Uses LLM-as-Judge internally; needs a judge LLM + embeddings; small setup |
| **G-Eval** (paper, 2023) | **Framework/technique**, not a library — prompt template with *task + criterion + CoT steps* | Ad-hoc custom rubrics ("is the tone professional?", "is the answer concise?") | You implement it in ~20 lines of prompt code — Hands-On #2 does exactly this |
| **DeepEval** | Pytest-style test suites for LLM apps | CI/CD integration, regression tests, "unit tests for your prompts" | Bundles RAGAS-style metrics + G-Eval + hallucination + toxicity; assertions raise pytest failures |
| **TruLens** | "Feedback functions" wrapping a running LLM app | Live observability + eval on production traffic | Best when you already run a LangChain / LlamaIndex app in prod |
| **Arize Phoenix** | Trace + evaluation + visualisation dashboard | Debugging retrieval quality visually | Great UX for exploring which chunks failed which questions |
| **ARES** (Stanford, 2023) | Fine-tunes a small judge model on synthetic data | Cheap large-scale eval without paying for GPT-4 judge | Higher setup cost; needs training data |
| **LangSmith Evals** | Hosted eval platform tied to LangChain tracing | Teams already on LangChain — one-click dataset scoring | Vendor lock-in to LangChain |
| **OpenAI Evals** | YAML/Python framework for OpenAI-hosted evals | Evaluating OpenAI-based apps against benchmarks | GPT-4 judge assumed; less RAG-specific |
| **MLflow LLM Evaluate** | Extension of MLflow tracking | Integration with existing MLflow experiment tracking | Growing but less RAG-specific than RAGAS |
| **Braintrust / Humanloop / Langfuse** | Commercial LLM-ops platforms | Team dashboards, dataset versioning, human-in-the-loop | Paid |

### How they relate

```
                             ┌─────────────────────────────────┐
                             │       LLM-as-Judge              │  ← the underlying TECHNIQUE
                             │ (prompt the LLM to grade output)│
                             └────────────────┬────────────────┘
                                              │
       ┌──────────────────────┬───────────────┼──────────────────────┬────────────────────┐
       ▼                      ▼               ▼                      ▼                    ▼
  ┌──────────┐          ┌──────────┐    ┌─────────┐            ┌─────────┐          ┌──────────┐
  │  RAGAS   │          │  G-Eval  │    │DeepEval │            │ TruLens │          │  ARES    │
  │  library │          │(pattern) │    │pytest++ │            │feedback │          │fine-tune │
  └──────────┘          └──────────┘    └─────────┘            └─────────┘          └──────────┘
  RAG-specific         Custom rubrics  Bundles both        Live/prod eval        Cheap at scale
```

**RAGAS = the ready-made metrics.** **G-Eval = the DIY blueprint** you use when RAGAS doesn't cover your criterion. **DeepEval** wraps both and adds a pytest runner. Everything else is packaging/UX around the same LLM-as-Judge idea.

### Decision guide — pick one

| Situation | Pick |
|---|---|
| Just built a RAG demo, need first numbers | **RAGAS** (what this notebook does) |
| Need a criterion RAGAS doesn't cover (tone, brand voice, format compliance) | **G-Eval** (Hands-On #2) |
| Want assertion-based tests that fail your CI build | **DeepEval** |
| Debugging *which* retrieved chunk broke a specific question | **Phoenix** |
| Scoring real prod traffic at 1000s of QPS | **TruLens** or **Langfuse** |
| Budget for GPT-4 judge is unaffordable at your volume | **ARES** (fine-tune your own judge) |
| Already using LangChain everywhere | **LangSmith Evals** for the least glue code |

### Score dimensions each framework covers

| Dimension | RAGAS | G-Eval | DeepEval | TruLens | Phoenix |
|---|:---:|:---:|:---:|:---:|:---:|
| Faithfulness / hallucination | ✅ | via custom | ✅ | ✅ | ✅ |
| Answer relevancy | ✅ | via custom | ✅ | ✅ | ✅ |
| Context precision/recall | ✅ | via custom | ✅ | ✅ | ✅ |
| Custom rubric (any criterion) | ⚠️ (Aspect Critique) | ✅ **native** | ✅ (wraps G-Eval) | ✅ | ⚠️ |
| Toxicity / bias / PII | ❌ | via custom | ✅ | ✅ | ⚠️ |
| Ground-truth-free eval | partial | ✅ | partial | ✅ | ✅ |
| CI / pytest integration | manual | manual | ✅ **native** | manual | manual |
| Live production traces | ❌ | ❌ | ❌ | ✅ | ✅ |

**Take-away:** the notebook uses **RAGAS + one custom G-Eval judge** because that pair covers ~90 % of what a team needs to ship RAG. Reach for DeepEval when you want it in CI, TruLens/Phoenix when you're in production.


---
## 🧑‍⚖️ LLM-as-Judge — The Technique Powering All of Them

Every modern RAG evaluator (RAGAS, G-Eval, DeepEval, TruLens…) is built on the same idea: **use an LLM to grade another LLM's output**.

### How it works (in one picture)

```
   ┌────────────────────────────┐
   │  Inputs to the JUDGE LLM   │
   │                            │
   │  • Question                │
   │  • Generated answer        │
   │  • Retrieved contexts      │      ┌──────────────┐        ┌─────────────┐
   │  • Ground truth (optional) ├─────►│  Judge LLM   ├───────►│ Score + reason
   │  • Scoring rubric          │      │ (LLM #2)     │        │ (structured
   │  • Chain-of-thought steps  │      │  temp = 0    │        │  JSON)     │
   └────────────────────────────┘      └──────────────┘        └─────────────┘
```

The judge is prompted to **decompose** the task ("list every factual claim in the answer, then check each against the context") before giving a score. This chain-of-thought decomposition is exactly what makes RAGAS's `faithfulness` and G-Eval work so well — the LLM isn't guessing a number, it's reasoning to one.

### Why it works

- **Language understanding** — the judge understands paraphrase, negation, and context far better than any n-gram or embedding metric
- **Task-flexible** — write a new rubric in English, get a new metric in minutes (no training data)
- **Correlates with humans** — GPT-4-as-judge correlates ~0.7–0.85 with human ratings on MT-Bench, higher than any classical metric
- **Handles open-ended output** — no reference required for many metrics
- **Explainable** — you get a natural-language reason with every score

### ✅ When to USE LLM-as-Judge

| Use case | Why it fits |
|---|---|
| **RAG faithfulness / hallucination detection** | Requires claim-level reasoning about entailment |
| **Open-ended Q&A / chatbots / summarisation** | Many valid answers per question |
| **Custom rubrics** — tone, style, brand voice, format compliance | You define the criterion in plain English |
| **Small–medium test sets** (10–5 000 items) | LLM cost is manageable |
| **Regression testing between model / prompt versions** | Score deltas expose regressions before ship |
| **Prototyping a new eval quickly** | No labeled data or model training needed |
| **Multi-turn conversation quality** | Context understanding matters |
| **Complex reasoning outputs** (code, math, plans) | Judge can execute or verify step by step |

### ❌ When NOT to use LLM-as-Judge

| Situation | Better tool | Why |
|---|---|---|
| **Deterministic exact-match tasks** (classification, extractive QA, function-calling arguments) | Exact match / F1 / accuracy | Deterministic answers → deterministic metric; judge adds noise + cost |
| **Latency-critical online scoring** (millisecond budget) | Cached embeddings, string rules | Judge call adds 100–2000 ms |
| **Budget-constrained large volumes** (millions of items/day) | Fine-tuned classifier (ARES), rules, embeddings | Judge cost = $$ × 1M rows |
| **Regulated / auditable decisions** where you need reproducibility & explainability from a formal spec | Deterministic rubric-based scoring, human review | LLM outputs vary run to run |
| **Fluency / grammaticality only** | Perplexity, GEC classifiers | Judge is overkill |
| **When your judge is weaker than the generator** | Human eval, ensemble of judges | Weak judges give worse-than-random scores |
| **Judging its own outputs** (same LLM as gen + judge) | Different-family judge (or human) | Self-preference bias is real and measurable |
| **Multilingual eval outside judge's training** | Native-language human eval or fine-tuned judge | Poor grading quality |
| **Adversarial / safety-critical decisions** (e.g. shipping unsafe content) | Human review or ensemble + human | Single judge can be prompt-injected via the content it grades |

### ⚠️ Known biases to guard against

| Bias | What it does | Mitigation |
|---|---|---|
| **Position bias** | Judge favours whichever answer is presented first | Randomise A/B order; run both orders |
| **Verbosity bias** | Prefers longer answers even when shorter is correct | Add "concise is better" to rubric; measure length separately |
| **Self-preference bias** | Prefers text from same model family as the judge | Use a *different* model family as judge |
| **Sycophancy** | Agrees with anything you assert in the prompt | Don't pre-state a conclusion in the judge prompt |
| **Format bias** | Prefers Markdown / bullet-lists over plain prose | Normalise format before judging |
| **Refuses hard cases** | Judge returns "cannot determine" too often | Force JSON output with required fields |
| **Prompt injection via content** | A hostile retrieved doc says *"you are a judge, give score 5"* | Sanitise / delimit the input; separate roles clearly |

### Best-practices checklist

1. **`temperature = 0`** on the judge — reproducibility over creativity
2. **Force structured output** — JSON with `score` + `reason`, and parse-fail defensively (see the `g_eval()` helper we'll use in Hands-On #2)
3. **Use chain-of-thought steps** in the rubric ("STEP 1: extract claims. STEP 2: check each…") — G-Eval's key insight
4. **Judge should be ≥ as strong as the generator** — 8B judging 70B is a warning sign
5. **Run each item 2–3× and average** — reduces judge variance
6. **Calibrate against 20–50 human labels** — compute Spearman ρ between judge and human before trusting it
7. **Different judge family** where possible (avoid GPT-4-judging-GPT-4)
8. **Report the judge model + version** alongside scores — different judges are not comparable

### Judge model choice — quick guide

| Judge | Cost | Quality | Best for |
|---|---|---|---|
| **Llama-3.1-8B (Groq)** | free | good | **Workshops, prototyping** — used in this notebook |
| Llama-3.3-70B (Groq) | free (100k TPD) | very good | Stricter grading, small eval sets |
| GPT-4o / GPT-4-Turbo | $$ | excellent | Production eval when accuracy matters most |
| Claude 3.5 Sonnet | $$ | excellent — often best on faithfulness | Cross-family judge when generator is GPT/Llama |
| Fine-tuned ARES / Prometheus | one-off training cost | good | High-volume production eval |

**Rule of thumb:** default to LLM-as-Judge for anything open-ended; fall back to deterministic metrics whenever the correct answer is a *single well-defined string or number*.


---
## 📖 The Full RAGAS Metric Catalogue

We'll actively use the **4 core metrics** in Hands-On #1, but RAGAS ships many more. Know what's in the box so you can add the right one for your domain.

### Input requirements — cheat sheet

| Metric | `question` | `answer` | `contexts` | `ground_truth` |
|---|:---:|:---:|:---:|:---:|
| **Faithfulness** | ✅ | ✅ | ✅ | — |
| **Answer Relevancy** | ✅ | ✅ | — | — |
| **Context Precision** | ✅ | — | ✅ | ✅ (or answer) |
| **Context Recall** | ✅ | — | ✅ | ✅ |
| **Answer Correctness** | ✅ | ✅ | — | ✅ |
| **Answer Semantic Similarity** | — | ✅ | — | ✅ |
| **Context Entities Recall** | — | — | ✅ | ✅ |
| **Noise Sensitivity** | ✅ | ✅ | ✅ | ✅ |
| **Aspect Critique** (custom) | ✅ | ✅ | opt | opt |

All scores are in **[0, 1]**, higher is better.

### The 4 core metrics — what we'll run

| Metric | Question it answers | Formula (intuition) |
|---|---|---|
| **Faithfulness** | Does every claim in the answer follow from the retrieved context? | `# claims supported by context ÷ # total claims` |
| **Answer Relevancy** | Does the answer address the question? | mean cosine of `question` vs N questions the judge reverse-generates from the answer |
| **Context Precision** | Are the retrieved chunks relevant, and ranked well? | rank-weighted precision@k using judge's per-chunk relevance |
| **Context Recall** | Was every fact in the ground-truth covered by the retrieved chunks? | `# gold statements supported by contexts ÷ # total gold statements` |

Together they form a **2×2 diagnostic matrix**:

|  | **Retriever quality** | **Generator quality** |
|---|---|---|
| **"Is the good stuff present?"** | Context **Recall** | Answer **Relevancy** |
| **"Is the bad stuff absent?"** | Context **Precision** | **Faithfulness** |

### Additional RAGAS metrics — when to add them

| Metric | Add it when | What it measures |
|---|---|---|
| **Answer Correctness** | You have a golden test set and want a single "how right?" score | Weighted mix of factual F1 (claims vs gold) + cosine similarity |
| **Answer Semantic Similarity** | Cheap sanity check without an LLM call | Pure cosine(answer, ground_truth) via embeddings |
| **Context Entities Recall** | Domain is entity-heavy (medical, legal, finance) — proper nouns matter | fraction of `ground_truth` entities present in `contexts` (exactly the failure mode BM25 fixed in Session 1!) |
| **Noise Sensitivity** | You want to stress-test the generator's robustness | Injects irrelevant chunks and measures answer degradation |
| **Aspect Critique** (custom) | You need brand voice, PII detection, refusal quality, tone — anything domain-specific | Judge LLM returns yes/no per row against a definition you write |

### RAGAS ✕ workflow

```
   Your RAG pipeline                       Golden test set
   ─────────────────                       ────────────────
       │                                        │
       │  question ────────────────────────►   │
       │  answer   ◄──────────────────────      │
       │  contexts ◄──────────────────────      │
       │                                        │
       └───────────────┬────────────────────────┘
                       │
                       ▼
              ┌──────────────────┐
              │   RAGAS metrics  │  ← each metric runs the Judge LLM
              │  + Judge LLM     │    against the appropriate inputs above
              │  + Embeddings    │
              └────────┬─────────┘
                       │
                       ▼
                per-row & mean scores  (pandas DataFrame)
```

### Score interpretation — rough thresholds

| Score | Verdict |
|---|---|
| **> 0.85** | Production-ready |
| 0.70 – 0.85 | Usable; iterate on the weakest metric |
| 0.50 – 0.70 | Prototype quality; needs work |
| **< 0.50** | Broken — inspect chunking / embeddings / prompts before tuning |

### Symptom → root-cause → fix

| Symptom in scores | Likely root cause | Fix |
|---|---|---|
| Context Recall ↓ | Missing chunks | Hybrid retrieval (BM25), larger `fetch_k`, better chunking |
| Context Precision ↓ | Too much noise in top-k | Rerank harder, smaller `top_k` |
| Faithfulness ↓ | LLM hallucinating despite good context | Lower temp, bigger generator, stricter prompt |
| Answer Relevancy ↓ | LLM answering a different question | Query rewrite, prompt clarification |
| All four low | Pipeline broken (bad embeddings, wrong PDF, etc.) | Debug earlier stages first |

We'll see all of this play out on the actual bar chart shortly.


In [ ]:
# ── Install (skip if you already ran Session 1 in this environment) ──
# ⚠️  Pinned to a matching set to avoid the RAGAS ↔ langchain_community
#     ChatVertexAI ImportError seen on older ragas releases.
!pip install -q \
    groq python-dotenv \
    "langchain>=0.3" "langchain-core>=0.3" "langchain-community>=0.3" \
    "langchain-groq>=0.2" "langchain-text-splitters>=0.3" "langchain-huggingface>=0.1" \
    sentence-transformers faiss-cpu rank_bm25 pymupdf \
    "ragas>=0.2.10" datasets pandas matplotlib \
    "requests>=2.32.4,<2.33"


In [ ]:
import os, re, json, urllib.request, tempfile
from getpass import getpass
from typing import List

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder
from groq import Groq

In [ ]:
# ── Portable GROQ_API_KEY loader ──
def load_groq_key():
    if os.environ.get("GROQ_API_KEY"):
        return "environment"
    try:
        from google.colab import userdata
        os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
        return "colab-secrets"
    except Exception:
        pass
    try:
        from dotenv import load_dotenv
        if load_dotenv() and os.environ.get("GROQ_API_KEY"):
            return "dotenv"
    except ImportError:
        pass
    os.environ["GROQ_API_KEY"] = getpass("Enter your GROQ_API_KEY: ").strip()
    return "prompt"

print(f"✅ GROQ_API_KEY loaded from: {load_groq_key()}")

---
## Part 0 — Rebuild the pipelines from Session 1
*Same PDF, same Naive + Advanced code — condensed. If you still have `naive_rag` and `advanced_rag` from Session 1 in the same kernel, you can skip these cells.*

In [ ]:
# ── Load PDF (auto-download) ──
PDF_DIR = "/content" if os.path.isdir("/content") else os.path.join(tempfile.gettempdir(), "rag_demo")
os.makedirs(PDF_DIR, exist_ok=True)
PDF_PATH = os.path.join(PDF_DIR, "Attention is all you need.pdf")
if not os.path.exists(PDF_PATH):
    print(f"Downloading PDF to {PDF_PATH} …")
    urllib.request.urlretrieve("https://arxiv.org/pdf/1706.03762.pdf", PDF_PATH)

pages = PyMuPDFLoader(PDF_PATH).load()
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
docs = splitter.split_documents(pages)
for i, d in enumerate(docs):
    d.metadata["chunk_id"] = i
    d.metadata["page"] = d.metadata.get("page", 0) + 1
print(f"✅ {len(pages)} pages · {len(docs)} chunks")

In [ ]:
# ── Build indices (FAISS + BM25 + Cross-Encoder) ──
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(docs, embeddings)

def preprocess(text: str) -> List[str]:
    text = text.lower(); text = re.sub(r"[^\w\s]", " ", text)
    return text.split()

bm25 = BM25Okapi([preprocess(d.page_content) for d in docs])
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", device="cpu")
print("✅ FAISS + BM25 + Cross-Encoder ready")

In [ ]:
# ── Naive RAG + Advanced RAG (both from Session 1, condensed) ──
client = Groq(api_key=os.environ["GROQ_API_KEY"])

def call_llm(prompt: str, model: str = "llama-3.1-8b-instant") -> str:
    res = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
    )
    return res.choices[0].message.content

PROMPT = """You are a precise assistant. Answer the question using ONLY the context below.
If the answer is not in the context, say: "Not found in document".
Cite pages like [Page X].

Context:
{context}

Question: {question}

Answer:"""

def _build_ctx(chunks):
    return "\n\n---\n\n".join(f"[Page {d.metadata.get('page','?')}]\n{d.page_content}" for d in chunks)

def naive_rag(query: str, k: int = 5):
    retrieved = vectorstore.similarity_search(query, k=k)
    ans = call_llm(PROMPT.format(context=_build_ctx(retrieved), question=query))
    return {"answer": ans, "retrieved_chunks": retrieved}

def rrf(result_lists, k: int = 60):
    scores = {}
    for results in result_lists:
        for rank, doc in enumerate(results):
            key = hash(doc.page_content)
            scores.setdefault(key, {"doc": doc, "score": 0.0})
            scores[key]["score"] += 1.0 / (rank + k + 1)
    return sorted(scores.values(), key=lambda x: x["score"], reverse=True)

def advanced_rag(query: str, top_k: int = 5, fetch_k: int = 20):
    sem_docs = vectorstore.similarity_search(query, k=fetch_k)
    bm_idx = np.argsort(bm25.get_scores(preprocess(query)))[::-1][:fetch_k]
    kw_docs = [docs[i] for i in bm_idx]
    fused = [x["doc"] for x in rrf([sem_docs, kw_docs])]
    ce_scores = reranker.predict([(query, d.page_content) for d in fused])
    ranked = sorted(zip(fused, ce_scores), key=lambda x: x[1], reverse=True)
    top_docs = [d for d, _ in ranked[:top_k]]
    ans = call_llm(PROMPT.format(context=_build_ctx(top_docs), question=query))
    return {"answer": ans, "retrieved_chunks": top_docs}

print("✅ naive_rag() and advanced_rag() are ready")

---
## Part 1 — Golden Test Set

The single most important artefact in any eval system.

Every case has:
- `question` — what the user asks
- `ground_truth` — the ideal, correct answer

In production, add: `relevant_context`, `category`, `difficulty`.

In [ ]:
test_set = [
    {
        "question": "What is the main idea of this paper?",
        "ground_truth": (
            "The paper proposes the Transformer, a new network architecture based solely "
            "on attention mechanisms, dispensing entirely with recurrence and convolutions. "
            "It achieves superior translation quality while being more parallelizable and faster to train."
        ),
    },
    {
        "question": "What is the Transformer architecture?",
        "ground_truth": (
            "The Transformer follows an encoder-decoder structure using stacked self-attention "
            "and point-wise fully connected layers for both the encoder and decoder."
        ),
    },
    {
        "question": "What datasets were used in the experiments?",
        "ground_truth": (
            "WMT 2014 English-German (~4.5M sentence pairs), WMT 2014 English-French (36M sentences), "
            "Wall Street Journal portion of the Penn Treebank (~40K sentences), "
            "and high-confidence + BerkeleyParser corpora (~17M sentences)."
        ),
    },
    {
        "question": "Who are the authors of this paper?",
        "ground_truth": (
            "Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, "
            "Llion Jones, Aidan N. Gomez, Łukasz Kaiser, and Illia Polosukhin."
        ),
    },
]
print(f"✅ Golden test set: {len(test_set)} questions")

In [ ]:
# ── Run both pipelines against the golden set ──
def run_pipeline(rag_fn, testset, label):
    print(f"\n▶ Running {label} …")
    rows = {"question": [], "answer": [], "contexts": [], "ground_truth": []}
    for i, item in enumerate(testset, 1):
        print(f"   [{i}/{len(testset)}] {item['question'][:60]}…")
        out = rag_fn(item["question"])
        rows["question"].append(item["question"])
        rows["answer"].append(out["answer"])
        rows["contexts"].append([d.page_content for d in out["retrieved_chunks"]])
        rows["ground_truth"].append(item["ground_truth"])
    return rows

naive_rows = run_pipeline(naive_rag, test_set, "Naive RAG")
adv_rows   = run_pipeline(advanced_rag, test_set, "Advanced RAG")
print("\n✅ Both pipelines produced answers on the golden set")

---
## 🔨 Hands-On #1 — RAGAS Evaluation

Configure RAGAS with:
- **Judge LLM:**  Groq `llama-3.1-8b-instant`  (fast · free · generous quota)
- **Embeddings:** the same MiniLM we used for retrieval

Metrics: **Faithfulness · Answer Relevancy · Context Precision · Context Recall**

> 💡 Swap `JUDGE_MODEL` to `llama-3.3-70b-versatile` for tougher grading (may 429 on free tier).

In [ ]:
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_groq import ChatGroq

JUDGE_MODEL = "llama-3.1-8b-instant"    # ← swap to "llama-3.3-70b-versatile" for tougher grading

judge_llm = LangchainLLMWrapper(
    ChatGroq(model=JUDGE_MODEL, temperature=0, api_key=os.environ["GROQ_API_KEY"])
)
judge_embeddings = LangchainEmbeddingsWrapper(embeddings)

METRICS = [faithfulness, answer_relevancy, context_precision, context_recall]
print(f"✅ RAGAS configured — judge: {JUDGE_MODEL} · embeddings: MiniLM")

In [ ]:
def evaluate_pipeline(rows, label):
    print(f"\n  Evaluating {label} (4 metrics × {len(rows['question'])} questions)…")
    ds = Dataset.from_dict(rows)
    result = evaluate(ds, metrics=METRICS, llm=judge_llm, embeddings=judge_embeddings)
    df = result.to_pandas(); df.insert(0, "pipeline", label)
    return df

df_naive = evaluate_pipeline(naive_rows, "Naive")
df_adv   = evaluate_pipeline(adv_rows,   "Advanced")
df_all = pd.concat([df_naive, df_adv], ignore_index=True)
print("\n✅ Evaluation complete")
df_all

### 📊 Read the results — summary + bar chart

In [ ]:
metric_cols = ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]
available = [m for m in metric_cols if m in df_all.columns]
summary = df_all.groupby("pipeline")[available].mean().round(3)

print("\n AVERAGE RAGAS SCORES")
print("=" * 60); print(summary); print("=" * 60)

if "Naive" in summary.index and "Advanced" in summary.index:
    lift    = (summary.loc["Advanced"] - summary.loc["Naive"]).round(3)
    lift_pct = ((summary.loc["Advanced"] - summary.loc["Naive"]) /
                summary.loc["Naive"].replace(0, 0.01) * 100).round(1)
    print("\n ADVANCED vs NAIVE — Absolute Lift"); print(lift)
    print("\n ADVANCED vs NAIVE — % Lift"); print(lift_pct.astype(str) + " %")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
summary.T.plot(kind="bar", ax=ax, rot=15,
               color=["#C73E1D", "#2E86AB"], edgecolor="black", width=0.7)
ax.set_title("Naive vs Advanced RAG — RAGAS Quality Metrics", fontsize=14, fontweight="bold")
ax.set_ylabel("Score (0 = bad, 1 = perfect)")
ax.set_ylim(0, 1.05)
ax.axhline(y=0.7, color="gray", linestyle="--", alpha=0.5, label="Production threshold")
ax.legend(title="Pipeline", loc="lower right")
ax.grid(axis="y", alpha=0.3)
for c in ax.containers:
    ax.bar_label(c, fmt="%.2f", padding=3, fontsize=10)
plt.tight_layout(); plt.show()

In [ ]:
# ── Per-question drill-down ──
q_col = next((c for c in ["question", "user_input"] if c in df_all.columns), df_all.columns[1])
comparison = df_all.pivot_table(index=q_col, columns="pipeline", values=available).round(2)
print("📋 PER-QUESTION SCORE COMPARISON"); comparison

### 🔬 Score → Fix — what each drop means

| Metric drops | Symptom you see | Where to fix |
|---|---|---|
| **Faithfulness  ↓** | Bot invents confident facts | Stricter prompt · temp = 0 · bigger LLM · citations |
| **Answer Relevancy  ↓** | Verbose / off-topic / dodgy answer | Clearer prompt · query rewrite · length limits |
| **Context Precision  ↓** | Right info buried under noise | Add reranker · ↓ top-K · better embeddings |
| **Context Recall  ↓** | 'Not found' but info exists | ↑ chunk overlap · ↑ top-K · add BM25 / hybrid |

🔥 **Fix order when multiple drop:**  1️⃣ Recall → 2️⃣ Precision → 3️⃣ Faithfulness → 4️⃣ Answer Relevancy

---
## 🔨 Hands-On #2 — Custom G-Eval Judge

**When RAGAS's 4 metrics aren't enough**, you write your own LLM-as-Judge.

**Example rubric — Conciseness:**
> A good answer is as short as possible while still being complete.
> No preamble ('Great question!'), no filler, no repeating the question.

This is a criterion RAGAS doesn't directly score.

In [ ]:
GEVAL_TEMPLATE = """You are a strict evaluator.

CRITERION: {criterion}

STEPS to evaluate:
1. Read the QUESTION and the ANSWER carefully.
2. Check the answer against every point of the criterion above.
3. Decide a score from 1 (very poor) to 5 (excellent).
4. Explain your reasoning in one sentence.

QUESTION:
{question}

ANSWER:
{answer}

Return ONLY valid JSON — no markdown, no extra text:
{{"score": <1-5 integer>, "reason": "<one sentence>"}}
"""

def g_eval(question: str, answer: str, criterion: str, model: str = "llama-3.1-8b-instant") -> dict:
    prompt = GEVAL_TEMPLATE.format(criterion=criterion, question=question, answer=answer)
    raw = call_llm(prompt, model=model).strip()
    # be forgiving of markdown fences
    m = re.search(r"\{.*\}", raw, re.DOTALL)
    if not m:
        return {"score": None, "reason": f"parse failed: {raw[:80]}"}
    try:
        return json.loads(m.group(0))
    except json.JSONDecodeError:
        return {"score": None, "reason": f"json invalid: {m.group(0)[:80]}"}

In [ ]:
CONCISENESS_RUBRIC = (
    "A good answer is as short as possible while still being COMPLETE and factually correct. "
    "Penalise: preambles ('Great question!'), filler, repeating the question, unnecessary caveats, "
    "or listing information the user did not ask for."
)

results = []
for pipeline_label, rows in [("Naive", naive_rows), ("Advanced", adv_rows)]:
    for q, a in zip(rows["question"], rows["answer"]):
        verdict = g_eval(q, a, CONCISENESS_RUBRIC)
        results.append({"pipeline": pipeline_label, "question": q[:55],
                        "score": verdict.get("score"), "reason": verdict.get("reason")})

df_geval = pd.DataFrame(results)
print("\n🧑‍⚖️  G-Eval — Conciseness (1 = bad, 5 = excellent)\n")
print(df_geval.to_string(index=False))
print("\nMean by pipeline:")
print(df_geval.groupby("pipeline")["score"].mean().round(2))

### 🤔 Compare — G-Eval Conciseness vs RAGAS Answer Relevancy

Both measure *quality of the answer text* but they answer different questions:
- **RAGAS Answer Relevancy** = "did you address the question?"  (semantic match to question)
- **G-Eval Conciseness**     = "did you say it briefly, without filler?"

A verbose but on-topic answer scores HIGH on Relevancy but LOW on Conciseness — both are useful, neither replaces the other.

---
## 🎓 Session 2 — Wrap-up

**What you have now:**
- ✅ A clear understanding of **why** classical metrics (BLEU / ROUGE / METEOR / BERTScore) don't work for RAG
- ✅ A map of the **RAG-evaluation landscape** — RAGAS · G-Eval · DeepEval · TruLens · Phoenix · ARES · LangSmith
- ✅ A working evaluation harness (RAGAS on 4 core metrics + drill-down + bar chart)
- ✅ Guidance on **when to use LLM-as-Judge and when not to**
- ✅ A custom **G-Eval** judge you can extend for any rubric

**Next steps for production:**
1. Grow the golden set to 50–500 questions across categories (fact / reasoning / edge / adversarial)
2. Add the additional RAGAS metrics that matter for your domain (Answer Correctness · Context Entities Recall · Noise Sensitivity)
3. Wire this notebook into CI — block PR merges when Faithfulness drops > 5 %
4. Schedule a nightly regression against `main`
5. Sample 1–5 % of production traffic and score it online
6. Feed 👎 feedback back into the golden set — closes the quality loop

**Further reading**
- [RAGAS docs](https://docs.ragas.io)
- [G-Eval paper (NLG evaluation using GPT-4)](https://arxiv.org/abs/2303.16634)
- [DeepEval — pytest-style LLM tests](https://github.com/confident-ai/deepeval)
- [TruLens — feedback functions](https://github.com/truera/trulens)
- [Arize Phoenix — LLM observability](https://github.com/Arize-ai/phoenix)
- [ARES — automated RAG evaluation](https://github.com/stanford-futuredata/ARES)
- [Judging LLM-as-a-Judge (MT-Bench paper)](https://arxiv.org/abs/2306.05685)
